# 1. Setup & Installs
Run this on Kaggle to install dependencies.


In [ ]:
!pip install transformers torch scikit-learn


# 2. Imports & Config
Includes the Kaggle dataset paths and Hyperparameter Grid.


In [ ]:
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import DistilBertTokenizerFast, DistilBertModel
import copy
from tqdm.auto import tqdm
import os
import itertools

# CONFIG
TRAIN_DATA_PATH = "/kaggle/input/datasets/raghavkapil/train-test-multi-head-distilbert/train_2000.json"
VAL_DATA_PATH = "/kaggle/input/datasets/raghavkapil/train-test-multi-head-distilbert/val_250.json"
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128
MAX_EPOCHS = 10
PATIENCE = 3  # Early stopping patience

# OPTIMIZED HYPERPARAMETER GRID (Target: 2-3 hours)
LEARNING_RATES = [1e-5, 2e-5, 5e-5]
BATCH_SIZES = [16, 32, 64]
WEIGHT_DECAYS = [0.01, 0.05, 0.1]
DROPOUT_RATES = [0.1, 0.2, 0.3]


# 3. Data Loading & Preprocessing
Maps our 3 outputs (complexity, rag, domain).


In [ ]:
with open(TRAIN_DATA_PATH, "r") as f:
    train_data = json.load(f)
with open(VAL_DATA_PATH, "r") as f:
    val_data = json.load(f)

domains = ["business", "coding", "creative_writing", "data_analysis", "education", "general", "mathematics", "reasoning", "science", "system_design"]
domain2id = {d: i for i, d in enumerate(domains)}

class MultiHeadDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        item = self.data[idx]
        text = item["prompt"]
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        label_complexity = 1.0 if item.get("label") == "strong" else 0.0
        label_rag = 1.0 if item.get("needs_rag") else 0.0
        label_domain = domain2id.get(item.get("domain", "general"), 5)
        
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "complexity": torch.tensor(label_complexity, dtype=torch.float),
            "rag": torch.tensor(label_rag, dtype=torch.float),
            "domain": torch.tensor(label_domain, dtype=torch.long)
        }

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
train_dataset = MultiHeadDataset(train_data, tokenizer, MAX_LEN)
val_dataset = MultiHeadDataset(val_data, tokenizer, MAX_LEN)


# 4. Multi-Head Model Architecture
Overrides DistilBert with adjustable Dropout.


In [ ]:
class MultiHeadRouter(nn.Module):
    def __init__(self, model_name, num_domains, dropout_rate=0.1):
        super().__init__()
        self.distilbert = DistilBertModel.from_pretrained(model_name)
        hidden_size = self.distilbert.config.hidden_size
        
        self.dropout = nn.Dropout(dropout_rate)
        self.complexity_head = nn.Linear(hidden_size, 1)
        self.rag_head = nn.Linear(hidden_size, 1)
        self.domain_head = nn.Linear(hidden_size, num_domains)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]
        pooled_output = self.dropout(pooled_output)
        
        out_complexity = self.complexity_head(pooled_output)
        out_rag = self.rag_head(pooled_output)
        out_domain = self.domain_head(pooled_output)
        
        return out_complexity, out_rag, out_domain


# 5. Evaluation Function
Calculates validation loss without updating weights.


In [ ]:
def evaluate(model, loader, device, loss_fn_complexity, loss_fn_rag, loss_fn_domain):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            b_complexity = batch["complexity"].to(device)
            b_rag = batch["rag"].to(device)
            b_domain = batch["domain"].to(device)
            
            out_comp, out_rag, out_dom = model(input_ids, attention_mask)
            
            loss_comp = loss_fn_complexity(out_comp.squeeze(), b_complexity)
            loss_rag = loss_fn_rag(out_rag.squeeze(), b_rag)
            loss_dom = loss_fn_domain(out_dom, b_domain)
            
            loss = loss_comp + loss_rag + loss_dom
            total_loss += loss.item()
            
    return total_loss / len(loader)


# 6. Hyperparameter Grid Search
Trains all combinations using itertools, with Early Stopping.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

ultimate_best_val_loss = float('inf')
ultimate_best_hyperparams = {}
ultimate_best_model_state = None

os.makedirs("router_model", exist_ok=True)
rag_pos_weight = torch.tensor([5.6]).to(device)

hyperparam_combos = list(itertools.product(BATCH_SIZES, LEARNING_RATES, WEIGHT_DECAYS, DROPOUT_RATES))
print(f"Testing {len(hyperparam_combos)} total configurations...")

for bs, lr, wd, dr in hyperparam_combos:
    print(f"\n========================================")
    print(f"🚀 Testing Config: Batch Size={bs}, LR={lr}, Weight Decay={wd}, Dropout={dr}")
    print(f"========================================")
    
    train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=bs)
    
    model = MultiHeadRouter(MODEL_NAME, len(domains), dropout_rate=dr).to(device)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=wd)
    
    loss_fn_complexity = nn.BCEWithLogitsLoss()
    loss_fn_rag = nn.BCEWithLogitsLoss(pos_weight=rag_pos_weight)
    loss_fn_domain = nn.CrossEntropyLoss()
    
    epochs_no_improve = 0
    best_combo_val_loss = float('inf')
    
    for epoch in range(MAX_EPOCHS):
        model.train()
        train_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS}", leave=False):
            optimizer.zero_grad()
            
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            b_complexity = batch["complexity"].to(device)
            b_rag = batch["rag"].to(device)
            b_domain = batch["domain"].to(device)
            
            out_comp, out_rag, out_dom = model(input_ids, attention_mask)
            loss = loss_fn_complexity(out_comp.squeeze(), b_complexity) + \
                   loss_fn_rag(out_rag.squeeze(), b_rag) + \
                   loss_fn_domain(out_dom, b_domain)
                   
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        train_loss /= len(train_loader)
        val_loss = evaluate(model, val_loader, device, loss_fn_complexity, loss_fn_rag, loss_fn_domain)
        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        
        # Check Early Stopping for this combo
        if val_loss < best_combo_val_loss:
            best_combo_val_loss = val_loss
            epochs_no_improve = 0
            
            # If this is the absolute best model we've ever seen across all combos, save it!
            if val_loss < ultimate_best_val_loss:
                ultimate_best_val_loss = val_loss
                ultimate_best_hyperparams = {'batch_size': bs, 'lr': lr, 'weight_decay': wd, 'dropout': dr, 'epoch': epoch+1}
                ultimate_best_model_state = copy.deepcopy(model.state_dict())
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"🛑 Early stopping triggered at epoch {epoch+1}! No improvement for {PATIENCE} epochs.")
                break

print(f"\n✅ Grid Search Complete!")
print(f"🏆 Ultimate Best Hyperparameters: {ultimate_best_hyperparams} with Val Loss: {ultimate_best_val_loss:.4f}")

torch.save(ultimate_best_model_state, "router_model/pytorch_model.bin")
tokenizer.save_pretrained("router_model")
print("Ultimate best model weights saved to router_model/!")

